## Interactive audio sample annotation

In [1]:
from __future__ import annotations  #Not needed for Python 3.10+
import yaml
from pathlib import Path
import matplotlib
import pandas as pd
from IPython.display import display #, HTML
import ipywidgets as widgets
button = widgets.Button(description="Continue")
output = widgets.Output()
import contextily as cx

from anqa.annotation import (MiniBirdNamer, FastMap, AnnotationState, load_labels,
                             normalize_secondary_labels, create_class_widgets,
                             SpectrogramAnnotator, AnnotationSession,
                             AnnotationControls, load_current_sample)

%matplotlib widget  
print(f'The Matplotlib backend is {matplotlib.get_backend()}')

The Matplotlib backend is widget


In [2]:
project_root = str(Path().resolve().parent.parent)
geographic_extents = {'new_zealand': {'min_longitude': 166,
                                        'max_longitude': 179,
                                        'min_latitude': -49,
                                        'max_latitude': -34},
                      }

In [3]:
try:
    yaml_path = Path(project_root) / 'project' / 'config.yaml'
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
        display(cfg)
except Exception as e:
    print("⚠️ There's a problem with config.yaml — check the file for typos or missing colons.")
    print(f"Details: {e}")
    raise

{'project_root': 'C:\\Users\\ollyp\\OneDrive\\Desktop\\anqa',
 'source_dataset_path': 'C:\\Users\\ollyp\\OneDrive\\Desktop\\anqa/data/demo_source',
 'reviewed_data_destn': 'C:\\Users\\ollyp\\OneDrive\\Desktop\\anqa/data/demo_source_reviewed',
 'audio_folder': 'C:\\Users\\ollyp\\OneDrive\\Desktop\\anqa/data/demo_source/audio',
 'naming_csv': 'C:\\Users\\ollyp\\OneDrive\\Desktop\\anqa/data/bird_names/bird_names.csv',
 'display_width': 16,
 'author': 'El Chupanibre',
 'reviewer': 'Alicanto',
 'region': 'new_zealand',
 'geographic_extents': {'new_zealand': {'min_latitude': -49.0,
   'max_latitude': -34.0,
   'min_longitude': 162.41,
   'max_longitude': -177.41},
  'south_america': {'min_longitude': -82.0,
   'max_longitude': -34.0,
   'min_latitude': -56.0,
   'max_latitude': 13.0}}}

In [4]:
source_dataset_path = cfg.get("source_dataset_path") or project_root + '/data/demo_source'
geographic_extents = cfg.get("geographic_extents") or geographic_extents
region = cfg.get('region') or 'new_zealand'
extents = geographic_extents[region]

use_case = {
    'project_root': project_root,
    'source_dataset_path': source_dataset_path,
    'reviewed_data_destn': cfg.get("reviewed_data_destn") or project_root + '/data/demo_reviewed',
    'naming_csv': cfg.get("naming_csv") or project_root + '/data/bird_names/bird_names.csv',
    'audio_folder': Path(source_dataset_path) / 'audio',
    'author': cfg["author"],
    'reviewer': cfg["reviewer"],
    'geographic_extents': geographic_extents,
    'region': region,
    'display_width': cfg["display_width"],
}

load_samples = False

In [5]:
class FilePaths:
    def __init__(self, options: dict):
        _project_dir = Path(options['project_root'])
        self.original_dataset = Path(options['source_dataset_path'])
        self.data_folder = _project_dir / 'data'
        self.basemap = self.data_folder / 'basemaps' / f"{options['region']}.png"
        self.audio_folder = options.get('audio_folder', self.data_folder)
        _metadata_files = [f for ext in ('csv', 'parquet') for f in self.original_dataset.glob(f'metadata.{ext}')]
        _label_files = [f for ext in ('csv', 'parquet') for f in self.original_dataset.glob(f'annotations.{ext}')]
        if len(_metadata_files) == 0:
            raise ValueError(f'No metadata.csv or metadata.parquet found in {self.original_dataset}')
        if len(_metadata_files) > 1:
            raise ValueError(f'Found both metadata.csv and metadata.parquet in {self.original_dataset} — ambiguous')
        if len(_label_files) > 1:
            raise ValueError(f'Found both metadata.csv and metadata.parquet in {self.original_dataset} — ambiguous')
        self.original_labels = _label_files[0] if len(_label_files) == 1 else None
        self.original_metadata = _metadata_files[0]
        self.out_dataset = Path(options['reviewed_data_destn'])
        self.out_dataset.mkdir(exist_ok=True, parents=True)
        self.out_labels = self.out_dataset / 'annotations.parquet'
        self.out_metadata = self.out_dataset / 'metadata.parquet'
        self.naming_csv = Path(options['naming_csv'])

In [6]:
paths = FilePaths(use_case)
namer = MiniBirdNamer(paths.naming_csv)

In [7]:
map = FastMap(map_extents=extents,
              basemap=paths.basemap,
              provider = cx.providers.Esri.WorldImagery) #fallback in case no png basemap found
annotation_state = AnnotationState(all_classes=namer.common_names, namer=namer, max_visible=30)

Start by loading the metadata dataframe, there should be exactly one metadata row per file

In [8]:
if paths.original_metadata.suffix == '.csv':
    df_meta = pd.read_csv(paths.original_metadata)
else:
    df_meta = pd.read_parquet(paths.original_metadata)

df_meta = df_meta.sort_values(by='filename')  #Ensures all the files from one class folder are presented sequentially
df_meta['secondary_labels'] = df_meta['secondary_labels'].apply(normalize_secondary_labels)
df_meta.head(3)

,filename,collection,secondary_labels,url,latitude,longitude,author,license,recorded_on,reviewed_by,reviewed_on,source_filename,source_start_s,source_end_s,models_used
30,20190831_074504_from_0.flac,NaN,[],NaN,-45.298842,176.298564,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,0.0,60.0,NaN
32,20190831_074504_from_120.flac,NaN,[],NaN,-42.175702,173.946154,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,120.0,180.0,NaN
33,20190831_074504_from_180.flac,NaN,[],NaN,-36.387599,166.194411,Sumudu,NaN,2019-08-31 07:45:04,NaN,NaN,20190831_074504.wav,180.0,240.0,NaN


Check previous labels if they exists:

In [9]:
df_labels = load_labels(paths.original_labels)
df_labels.head()

,Filename,Start Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),Label,Type,Sex,Score,Delta Time (s),Delta Freq (Hz),Avg Power Density (dB FS/Hz),Life Stage,Indv ID
0,20190831_083004_from_420.flac,37.6,45.0,1230.0,2365.0,weka1,<NA>,<NA>,NaN,7.4,1135.0,-53.0,<NA>,<NA>
1,20190831_181504_from_240.flac,54.5,60.0,997.0,2656.0,weka1,<NA>,<NA>,NaN,5.5,1659.0,-62.1,<NA>,<NA>
2,20190831_181504_from_300.flac,0.0,5.1,997.0,2656.0,weka1,<NA>,<NA>,NaN,5.1,1659.0,-61.2,<NA>,<NA>
3,20190831_181504_from_300.flac,10.3,37.8,1133.0,2655.0,weka1,<NA>,<NA>,NaN,27.5,1522.0,-62.2,<NA>,<NA>
4,20190831_181504_from_780.flac,13.4,49.5,444.0,8302.0,weka1,<NA>,<NA>,NaN,36.1,7858.0,-45.6,<NA>,<NA>


Check the naming schema

In [10]:
df_naming = pd.read_csv(paths.naming_csv)
df_naming.head(3)

,CommonName,eBird,ScientificName,ClassifyName,AnnotationGroup,OrderName
0,Northern Royal Albatross,norroal,Diomedea sanfordi,Northern Royal Albatross,Birds,Albatross
1,Bellbird,nezbel1,Anthornis melanura,Bellbird,Birds,Bellbird
2,Bellbird or Tui,bellbird_tui,Anthornis melanura or Prosthemadera novaeseela...,Bellbird-Tui,Fallback,Bellbird


## Annotation
* **Every** Bird or Animal Sound to be boxed
* Unknown classes to be labelled *Unknown*
* Calls from the same bird with gaps of greater than 2 seconds should have individual boxes
* Otherwise a single large box should be used
* See below for the keyboard shortcuts

In [11]:
annotation_groups = dict(zip(df_naming['CommonName'], df_naming['AnnotationGroup']))
create_class_widgets(annotation_state,
                     n_columns=6,
                     fastmap=map,
                     common_to_ebird=namer.common_to_ebird_dict,
                     class_groups=annotation_groups,
                     group_order=[])

annotator = SpectrogramAnnotator(annotation_state,
                                 common_to_ebird=namer.common_to_ebird_dict,
                                 plot_size = (use_case['display_width'],4),  #Adjust for screen size
                                 f_min=20,
                                 f_max=16000,
                                 zoom_window_height=0.4,
                                 zoom_window_width=5,
                                 min_drag_rows=5,
                                 min_drag_time_s=.1,
                                 min_separation = 2,
                                 similarness_threshold=0.5,
                                 min_freq_hz=300)    ####################Not working yet #########################

session = AnnotationSession(df_meta=df_meta,
                            df_labels=df_labels,
                            new_meta_filepath=paths.out_metadata,
                            new_labels_filepath=paths.out_labels,
                            reviewer = use_case['reviewer'],
                            author = use_case['author'])
controls = AnnotationControls()
annotation_widget = load_current_sample(session, annotator, paths, map)
controls.display()
controls.bind(session, annotator, paths, map)

def on_space_key(event):
    if event.key == ' ':
        controls._on_next_clicked()

annotator.fig.canvas.mpl_connect('key_press_event', on_space_key)

21

## Shortcuts

| Action | Result |
|--------| ---------------- |
| **Left mouse click-drag** |Starts box drawing on click, finishes on release |
| **Left mouse click** |Repeats previous box, but centred on the pointer|
| **Right mouse click** | Moves the zoom box and restarts the playback at that point|
| **Space Bar** | Saves current file and moves on to the next one |
| **u** | Undoes the last box |
| **<--     -->**| Cycles through existing boxes to highlight the one to undo |
| **d** | Deletes all boxes (including originals) |
| **t** | Tries to box any identical patterns from the last un the same frequency limits |
| **b** | Tries to mark calls based on power peaks.  Not recommended, needs improvement |
| **g** | Places a grid of 10 second spacing |
| **g again** | A vertical and horizontal grid |
| **g again** | No grid |

### Progress Checks

In [12]:
session.summary()

{'total_files': 45,
 'finished_files_in_new_meta': 6,
 'total_annotations': 71,
 'done_in_current_session': np.int64(6),
 'pending_in_current_session': np.int64(39),
 'total_minus_pending': np.int64(6)}

In [13]:
marked_times = annotator.get_boxes()
marked_times[-1:]

[]

If there were previous sessions, an example of their most recent saved data is shown below

In [14]:
if paths.out_metadata.exists():
    output_metadata = pd.read_parquet(paths.out_metadata)
    display(output_metadata.tail())

,filename,collection,primary_label,secondary_labels,url,latitude,longitude,author,license,recorded_on,reviewed_by,reviewed_on,source_filename,source_sr_khz,source_start_s,source_end_s,source_device,models_used
1,20190831_074504_from_120.flac,<NA>,<NA>,[],<NA>,-42.175702,173.946154,Sumudu,<NA>,2019-08-31 07:45:04,<NA>,NaN,20190831_074504.wav,NaN,120.0,180.0,<NA>,<NA>
2,20190831_074504_from_180.flac,<NA>,<NA>,[],<NA>,-36.387599,166.194411,Sumudu,<NA>,2019-08-31 07:45:04,<NA>,NaN,20190831_074504.wav,NaN,180.0,240.0,<NA>,<NA>
3,20190831_074504_from_240.flac,<NA>,<NA>,[],<NA>,-38.853367,170.015270,Sumudu,<NA>,2019-08-31 07:45:04,<NA>,NaN,20190831_074504.wav,NaN,240.0,300.0,<NA>,<NA>
4,20190831_074504_from_300.flac,<NA>,<NA>,[],<NA>,-44.351759,169.393210,Sumudu,<NA>,2019-08-31 07:45:04,<NA>,NaN,20190831_074504.wav,NaN,300.0,360.0,<NA>,<NA>
5,20190831_074504_from_360.flac,<NA>,<NA>,[],<NA>,-34.054730,175.899038,El Chupanibre,<NA>,2019-08-31 07:45:04,Alicanto,2026-09-17,20190831_074504.wav,NaN,360.0,420.0,<NA>,<NA>


In [15]:
if paths.out_metadata.exists():
    display(output_metadata.shape)

(6, 18)

In [16]:
if paths.out_labels.exists():
    output_labeldata = pd.read_parquet(paths.out_labels)
    display(output_labeldata.tail())

,Filename,Start Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),Label,Type,Sex,Score,Life Stage,Indv ID,Delta Time (s),Delta Freq (Hz),Avg Power Density (dB FS/Hz)
49,20190831_074504_from_240.flac,34.5,37.5,8012.0,11055.0,riflem1,None,None,None,None,None,3.0,3043.0,-60.7
50,20190831_074504_from_240.flac,43.6,44.4,8103.0,10599.0,riflem1,None,None,None,None,None,0.8,2496.0,-57.9
51,20190831_074504_from_240.flac,47.0,47.5,9199.0,11329.0,riflem1,None,None,None,None,None,0.5,2130.0,-47.2
52,20190831_074504_from_240.flac,53.0,53.4,8407.0,10568.0,riflem1,None,None,None,None,None,0.4,2161.0,-46.9
53,20190831_074504_from_300.flac,7.3,8.9,7616.0,11025.0,riflem1,None,None,None,None,None,1.6,3409.0,-46.9
